In [1]:
import pandas as pd
import os
os.environ['TORCH_USE_CUDA_DSA'] = '1'
import torch
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from langchain.document_loaders import PyMuPDFLoader
from langchain.vectorstores import Chroma
import re
import time
import ast

C:\Users\Jacky\AppData\Roaming\Python\Python39\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [2]:
embedding_model = SentenceTransformer("Snowflake/snowflake-arctic-embed-l-v2.0", trust_remote_code=True,token=os.getenv('HF_TOKEN'))

In [3]:
def clean_text(text):
    cleaned = re.sub(r'[-()"#/@;:<>{}=\-~|.?,]', ' ', text)
    return cleaned.strip()

In [4]:
from langchain.schema import Document
pdf_directory = "datasets"
pdf_names = [pdf_name for pdf_name in os.listdir(pdf_directory) if pdf_name.endswith('.pdf')]
split_documents = []
for pdf_name in tqdm(pdf_names, desc="Loading PDFs"):
    pdf_path = os.path.join(pdf_directory, pdf_name)
    loader = PyMuPDFLoader(pdf_path)
    loaded_documents = loader.load()
    for document in loaded_documents:
        split_texts = clean_text(document.page_content)
        split_documents.append(Document(page_content=split_texts, metadata=document.metadata))

Loading PDFs: 100%|██████████| 9/9 [00:01<00:00,  4.88it/s]


In [5]:
from chromadb.api.types import Documents, EmbeddingFunction, Embeddings
class MyEmbeddingFunction(EmbeddingFunction):
    def __call__(self, texts: Documents) -> Embeddings:
        embeddings = [embedding_model.encode(x) for x in texts]
        return embeddings

In [6]:
import chromadb
chroma_client = chromadb.Client()

collection = chroma_client.create_collection(metadata={"hnsw:space": "cosine"},name="my_collection",embedding_function=MyEmbeddingFunction())

collection.add(
    documents=[doc.page_content for doc in split_documents],
    metadatas=[doc.metadata for doc in split_documents],
    ids=[f"id{index+1}" for index in range(len(split_documents))]
)

KeyboardInterrupt: 

In [7]:
question_df = pd.read_excel('test.xlsx')

In [8]:
all_retrieved_pdf = []
all_pdf_page = []
all_content = []
all_distance = []
for query in question_df['query']:
    retrieved_pdf = []
    pdf_page = []
    context = []
    distance = []
    results = collection.query(query_texts=query,n_results=3)
    for i in range(3):
        retrieved_pdf.append(results['metadatas'][0][i]['source'].split('datasets\\')[-1].split('.pdf')[0])
        pdf_page.append(results['metadatas'][0][i]['page'] + 1)
        context.append(results['documents'][0][i])
        distance.append(results['distances'][0][i])
    all_retrieved_pdf.append(retrieved_pdf)
    all_pdf_page.append(pdf_page)
    all_content.append(context)
    all_distance.append(distance)

In [9]:
question_df = question_df.drop(['Unnamed: 0'],axis=1)

In [10]:
all_retrieved_pdf_df = pd.DataFrame(all_retrieved_pdf, columns=['pdf_1', 'pdf_2', 'pdf_3'])
all_pdf_page_df = pd.DataFrame(all_pdf_page, columns=['pdf_page_1', 'pdf_page_2', 'pdf_page_3'])
all_content_df = pd.DataFrame(all_content, columns=['context_1', 'context_2', 'context_3'])
all_distance_df = pd.DataFrame(all_distance, columns=['distance_1', 'distance_2', 'distance_3'])

final_df = pd.concat([question_df, all_retrieved_pdf_df],axis = 1)
final_df = pd.concat([final_df, all_pdf_page_df],axis = 1)
final_df = pd.concat([final_df, all_content_df],axis = 1)
final_df = pd.concat([final_df, all_distance_df],axis = 1)

In [11]:
final_df.to_excel('chroma_gte.xlsx')

In [5]:
vector_store = Chroma.from_documents(split_documents, embedding_model)
retriever=vector_store.as_retriever(search_kwargs={"k": 3})

In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from accelerate import Accelerator
accelerator = Accelerator()
model_name="google/gemma-2-9b-it"
cache_dir = "/mnt/harddisk/cache"
os.makedirs(cache_dir, exist_ok=True)
tokenizer = AutoTokenizer.from_pretrained(model_name,cache_dir=cache_dir,device_map="auto",torch_dtype=torch.bfloat16, token=os.getenv('HF_TOKEN'))
model = AutoModelForCausalLM.from_pretrained(model_name,cache_dir=cache_dir,device_map="auto",torch_dtype=torch.bfloat16,ignore_mismatched_sizes=True, token=os.getenv('HF_TOKEN'))

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [20]:
results=vector_store.asimilarity_search_with_relevance_scores(query)

In [3]:
def prompt_rag(context,question):
    return f"""
    使用所給的上下文逐步推理來回答問題，並且可以使用已有的知識。
    使用不超過三句來回答，並且用繁體中文。
    上下文: {context}
    Question: {question}
    Answer:
    """
def prompt_no_rag(question):
    return f"""
    根據下列問題，逐步推理並回答，並且用繁體中文。
    使用不超過三句來回答，並且用繁體中文。
    Question: {question}
    Answer:
    """
def get_chatbot_response(user_message):
    # Retrieve relevant documents
    retrieved_docs = retriever.get_relevant_documents(user_message)
    contexts = []
    for context in retrieved_docs:
        contexts.append(context.page_content)
    # Prepare the input for the language model
    prompt = prompt_rag(contexts,user_message)
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = inputs.to('cuda')
    input_token = len(tokenizer(prompt)["input_ids"])
    outputs = model.generate(**inputs,max_length=input_token+100)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = response.split('Answer:\n')[1]
    return answer

In [4]:
import string
def normalize_text(s):
    def remove_char_num(text):
        return re.sub(r'[a-zA-Z]', ' ', text)

    def white_space_fix(text):
        return ' '.join(text.split())

    def remove_punc(text):
        exclude = set(string.punctuation)
        return ''.join(ch for ch in text if ch not in exclude)
        
    def change_enter(text):
        return re.sub(r'\n|\\n', '。', text)

    return white_space_fix(remove_char_num(remove_punc(change_enter(s))))

In [5]:
def self_reflective_prompt(query, responses, contexts):
    return f"""
    您被要求根據其相關性和支持性評估多個回應段落，這些評估是針對用戶查詢和檢索到的文檔。

    ISREL: 評估檢索到的上下文是否與用戶查詢相關。回答選項為「是」或「否」。
    ISUSE: 評估檢索到的文檔是否在生成回應中被使用。回答選項為「是」或「否」。
    ISSUP: 評估回應段落是否得到檢索文檔的支持。如果回應直接引用文檔中的信息或與其內容一致，則被視為支持。回答選項為「是」或「否」。

    請生成回應進行評估， 填入適合的ISREL, ISUSE, ISSUP回答選項。
    用戶查詢: {query}
    檢索上下文: {contexts}
    生成回應: {responses}
    答案:
    ISREL:
    ISUSE:
    ISSUP:
    """

In [7]:
import ast
answers = []
segments = []
input_tokens = []
output_tokens = []
retrieved_contexts = question__with_retrieved['retrieved_context']
question = question__with_retrieved['query']
for q, retrieved_context in tqdm(zip(question, retrieved_contexts)):
    max_score = -1
    answer = ""
    segment = []
    all_input_token = []
    all_output_token = []
    for r in ast.literal_eval(retrieved_context):
        r = normalize_text(r)
        prompt = prompt_rag(r,q)
        inputs = tokenizer(prompt, return_tensors="pt").to('cuda')
        input_token = len(tokenizer(prompt)["input_ids"])
        all_input_token.append(input_token)
        outputs = model.generate(**inputs,max_length=input_token+200)
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        model_response = normalize_text(response.split('Answer:\n')[1])
        output_token = len(tokenizer(model_response)["input_ids"])
        all_output_token.append(output_token)
        segment.append(model_response)
    input_tokens.append(all_input_token)
    output_tokens.append(all_output_token)
    segments.append(segment)

0it [00:00, ?it/s]The 'max_batch_size' argument of HybridCache is deprecated and will be removed in v4.46. Use the more precisely named 'batch_size' argument instead.
Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)
2025-02-09 18:57:48.399200: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1739127468.436703  202498 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1739127468.442873  202498 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-09 18:57:48.473706: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available

In [ ]:
answers = []
segments = []
input_tokens = []
output_tokens = []
retrieved_contexts = question__with_retrieved['retrieved_context']
question = question__with_retrieved['query']
for q, retrieved_context in tqdm(zip(question, retrieved_contexts)):
    max_score = -1
    answer = ""
    segment = []
    all_input_token = []
    all_output_token = []
    for r in ast.literal_eval(retrieved_context):
        r = normalize_text(r)
        prompt = prompt_rag(r,q)
        inputs = processor(text = prompt,add_special_tokens=False,return_tensors="pt").to(model.device)
        input_token = len(inputs['input_ids'][0])
        all_input_token.append(input_token)
        output = model.generate(**inputs, max_new_tokens=200)
        response = processor.decode(output[0])
        model_response = normalize_text(response.split('Answer:\n')[1])
        output_token = len(output[0]) - input_token
        all_output_token.append(output_token)
        segment.append(model_response)
    input_tokens.append(all_input_token)
    output_tokens.append(all_output_token)
    segments.append(segment)

In [9]:
input_token_df = pd.DataFrame(input_tokens, columns=['input_token_1', 'input_token_2', 'input_token_3'])
output_token_df = pd.DataFrame(output_tokens, columns=['output_token_1', 'output_token_2', 'output_token_3'])
segments_df = pd.DataFrame(segments, columns=['retrieve_context_1', 'retrieve_context_2', 'retrieve_context_3'])


final_df = pd.concat([question__with_retrieved, input_token_df],axis = 1)
final_df = pd.concat([final_df, output_token_df],axis = 1)
final_df = pd.concat([final_df, segments_df],axis = 1)

In [10]:
final_df.to_excel('Result/stage_2/self_rag/llama_11b_instruct_self_rag_segments_010446.xlsx')

In [6]:
self_rag_segment = pd.read_excel('Result/stage_2/self_rag/llama_11b_instruct_self_rag_segments_010446.xlsx')

In [7]:
self_rag_segment.rename(columns={
    'retrieve_context_1': 'segment_1',
    'retrieve_context_2': 'segment_2',
    'retrieve_context_3': 'segment_3'
}, inplace=True)
self_rag_segment.head(1)

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,pdf,query,answer,relevant context,Unnamed: 5,retrieved_context,input_token_1,input_token_2,input_token_3,output_token_1,output_token_2,output_token_3,segment_1,segment_2,segment_3
0,0,0,1,GS-1,TCFD在風險管理上應有甚麼建議?,根據氣候相關財務揭露(TCFD)的建議，組織應描述組織辨識和評估氣候相關風險的流程，描述組織...,"Disclose how the organisation identifies, asse...",NaN,['監管政策手冊\nTM-G-1\n科技風險管理的㆒般原則\nV.1 – 24.06.03\...,296,527,448,200,200,200,根據監管政策手冊，科技風險管理的基本原則包括了 （金融穩定調查委員會）的建議。 建議在風險管...,根據 的建議，風險管理應該是全面和可持续的，包括识别、评估和管理各种风险，包括气候风险、市场...,建議企業應透過預防、補償及應變措施來管控風險。企業應識別漸現或現有的科技有關風險，包括系統發...


In [14]:
max_scores = []
final_response = []
judge = []
judge_score = []
for n in tqdm(range(len(self_rag_segment))):
    max_score = 0
    target = ""
    question = self_rag_segment['query'][n]
    retrieved_context = ast.literal_eval(self_rag_segment['retrieved_context'][n])
    answers = [self_rag_segment['segment_1'][n],self_rag_segment['segment_2'][n],self_rag_segment['segment_3'][n]]
    all_response = []
    all_score = []
    for i in range(3):
        q = question
        a = answers
        r = retrieved_context
        self_prompt = self_reflective_prompt(q,a[i], r[i])
        inputs = tokenizer(self_prompt, return_tensors="pt").to('cuda')
        input_token = len(tokenizer(self_prompt)["input_ids"])
        outputs = model.generate(**inputs,max_length=input_token+100)
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        all_response.append(response)
        score = 0
        score += sum('是' in response.split(key)[-1][:5] for key in ['ISREL:', 'ISUSE:', 'ISSUP:'])
        all_score.append(score)
        max_score, target = (score, a[i]) if score > max_score else (max_score, target)
    max_scores.append(max_score)
    final_response.append(target)
    judge.append(all_response)
    judge_score.append(all_score)

100%|███████████████████████████████████████████████████████████████████████████████████| 90/90 [18:07<00:00, 12.09s/it]


In [9]:
max_scores = []
final_response = []
judge = []
judge_score = []
for n in tqdm(range(len(self_rag_segment))):
    max_score = 0
    target = ""
    question = self_rag_segment['query'][n]
    retrieved_context = ast.literal_eval(self_rag_segment['retrieved_context'][n])
    answers = [self_rag_segment['segment_1'][n],self_rag_segment['segment_2'][n],self_rag_segment['segment_3'][n]]
    all_response = []
    all_score = []
    for i in range(3):
        q = question
        a = answers
        r = retrieved_context
        self_prompt = self_reflective_prompt(q,a[i], r[i])
        inputs = processor(text = self_prompt,add_special_tokens=False,return_tensors="pt").to(model.device)
        input_token = len(inputs['input_ids'][0])
        output = model.generate(**inputs, max_new_tokens=200)
        response = processor.decode(output[0])
        all_response.append(response)
        score = 0
        score += sum('是' in response.split(key)[-1][:5] for key in ['ISREL:', 'ISUSE:', 'ISSUP:'])
        all_score.append(score)
        max_score, target = (score, a[i]) if score > max_score else (max_score, target)
    max_scores.append(max_score)
    final_response.append(target)
    judge.append(all_response)
    judge_score.append(all_score)

100%|█████████████████████████████████████████████████████████████████████████████████| 90/90 [1:16:02<00:00, 50.70s/it]


In [10]:
self_rag_segment['max_score'] = max_scores
self_rag_segment['model_response'] = final_response
judge_df = pd.DataFrame(judge, columns=['judge_1', 'judge_2', 'judge_3'])
judge_score_df = pd.DataFrame(judge_score, columns=['judge_score_1', 'judge_score_2', 'judge_score_3'])

final_df = pd.concat([self_rag_segment, judge_df],axis = 1)
final_df = pd.concat([final_df, judge_score_df],axis = 1)

In [11]:
final_df.to_excel('Result/stage_2/self_rag/llama_11b_instruct_self_rag_.xlsx')

In [60]:
sum(1 for key in ['ISREL:', 'ISUSE:', 'ISSUP:'] if '是' in response.split(key)[-1][:5])

0

In [16]:
question_df['query'][0]

'TCFD在風險管理上應有甚麼建議?'

In [4]:
def rag_once(prompt):
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = inputs.to('cuda')
    input_token = len(tokenizer(prompt)["input_ids"])
    outputs = model.generate(**inputs,max_length=input_token+200)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = response.split('Answer:\n')[1]
    output_token = len(tokenizer(answer)["input_ids"])
    return input_token, output_token, answer
def rag_once_llama(prompt):
    inputs = processor(
            text = prompt,
            add_special_tokens=False,
            return_tensors="pt"
        ).to(model.device)
    output = model.generate(**inputs, max_new_tokens=200)
    response = processor.decode(output[0])
    answer = response.split('Answer:')[1].split(']]>')[0]
    input_token = len(inputs['input_ids'][0])
    output_token = len(output[0]) - input_token
    return input_token, output_token, answer

In [6]:
question_df.columns

Index(['Unnamed: 0', 'pdf', 'query', 'answer', 'relevant context',
       'Unnamed: 5'],
      dtype='object')

In [6]:
input_tokens = []
output_tokens = []
answers = []
for q in tqdm(question_df['query']):
    prompt = prompt_no_rag(q)
    input_token, output_token, answer = rag_once(prompt)
    input_tokens.append(input_token)
    output_tokens.append(output_token)
    answers.append(answer)
    torch.cuda.empty_cache()

100%|███████████████████████████████████████████████████████████████████████████████████| 90/90 [23:27<00:00, 15.64s/it]


In [8]:
question_df['input_token'] = input_tokens
question_df['output_token'] = output_tokens
question_df['model_response'] = answers

In [9]:
question_df.to_excel('Result/stage_2/no_rag/yi1.5_9b_chat_no_rag_2327.xlsx')

In [25]:
retrieved_context = []
for q in tqdm(question_df['query']):
    retrieved_docs = retriever.get_relevant_documents(q)
    contexts = [context.page_content for context in retrieved_docs]
    retrieved_context.append(contexts)
question_df['retrieved_context'] = retrieved_context

100%|███████████████████████████████████████████████████████████████████████████████████| 90/90 [00:02<00:00, 44.55it/s]


In [26]:
question_df.to_excel('test_with_retrieved.xlsx')

In [12]:
input_tokens = []
output_tokens = []
answers = []
for index, row in tqdm(question__with_retrieved[['query', 'retrieved_context']].iterrows()):
    q = row['query']
    r = row['retrieved_context']
    prompt = prompt_rag(r, q)
    input_token, output_token, answer = rag_once(prompt)
    input_tokens.append(input_token)
    output_tokens.append(output_token)
    answers.append(answer)

90it [42:34, 28.38s/it]


In [13]:
question__with_retrieved['input_token'] = input_tokens
question__with_retrieved['output_token'] = output_tokens
question__with_retrieved['model_response'] = answers

In [15]:
question__with_retrieved.to_excel('Result/stage_2/simple_rag/yi1.5_9b_chat_rag_4234.xlsx')

In [38]:
def ircot_llama(question):
    retrieved_context = []
    reason = question
    maxi_iter = 3
    for i in range(maxi_iter):
        retrieved_docs = retriever.get_relevant_documents(reason)
        context = retrieved_docs[0].page_content
        context = clean_text(context)
        if context not in retrieved_context:
            retrieved_context.append(context)
        prompt = prompt_ircot(retrieved_context, question)
        inputs = processor(
                text = prompt,
                add_special_tokens=False,
                return_tensors="pt"
            ).to(model.device)
        input_token = len(inputs['input_ids'][0])
        outputs = model.generate(**inputs, max_new_tokens=200)
        response = processor.decode(outputs[0])
        answer = response.split('Answer:\n')[1]
        output_token = len(outputs[0]) - input_token
        reason = answer
    return answer, retrieved_context, input_token, output_token

In [13]:
def ircot(question):
    retrieved_context = []
    reason = question
    maxi_iter = 3
    for i in range(maxi_iter):
        retrieved_docs = retriever.get_relevant_documents(reason)
        context = retrieved_docs[0].page_content
        context = clean_text(context)
        if context not in retrieved_context:
            retrieved_context.append(context)
        prompt = prompt_ircot(retrieved_context, question)
        inputs = tokenizer(prompt, return_tensors="pt")
        inputs = inputs.to('cuda')
        input_token = len(tokenizer(prompt)["input_ids"])
        outputs = model.generate(**inputs,max_length=input_token+100)
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        answer = response.split('Answer:\n')[1]
        output_token = len(tokenizer(answer)["input_ids"])
        reason = answer
    return answer, retrieved_context, input_token, output_token

In [39]:
answers, retrieved_contexts, input_tokens, output_tokens = [], [], [], []
for q in tqdm(question_df['query']):
    answer, retrieved_context, input_token, output_token = ircot_llama(q)
    answers.append(answer)
    retrieved_contexts.append(retrieved_context)
    input_tokens.append(input_token)
    output_tokens.append(output_token)

100%|███████████████████████████████████████████████████████████████████████████████████| 27/27 [20:35<00:00, 45.76s/it]


In [41]:
question_df['input_token'] = input_tokens
question_df['output_token'] = output_tokens
question_df['first_answer'] = answers
question_df['retrieved_context'] = retrieved_contexts

In [17]:
question_df.to_excel('ircot_llama_1433.xlsx')

In [58]:
if context not in retrieved_contexts[20]:
    print('yes')

In [4]:
context = pd.read_excel('retrived_text.xlsx')

In [26]:
prompt = prompt_rag('''電傳轉帳是由一間機構(匯款機構)代表某人(匯款人)藉電 子方式進行的交易，目的是將某筆金錢轉往某間機構(收 款機構) (該機構可以是匯款機構或另一機構)以提供予 該人或另一人(收款人)，而無論是否有一間或多於一間 其他機構(中介機構)參與完成有關金錢轉帳。''','電傳轉帳是甚麼?')

In [27]:
inputs = tokenizer(prompt, return_tensors="pt")
inputs = inputs.to('cuda')
input_token = len(tokenizer(prompt)["input_ids"])
outputs = model.generate(**inputs,max_length=input_token+100)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
answer = response.split('Answer:\n')[1]

In [28]:
answer

'    電傳轉帳是一種透過電子方式，由匯款機構代表匯款人將金錢轉至收款機構，提供給收款人或另一人，無論是否透過中介機構完成的交易。\n\n\n'

In [51]:
def critique_responses(query, contexts, responses):
    critique_prompt = f"""
    You are tasked with evaluating the quality of multiple response segments based on their relevance and supportiveness 
    with respect to the user query and retrieved documents.

    ISUSE: Assess whether the retrieved document was used in generating the response. This means the response relied on the document to provide 
    its content. Answer with "Yes" or "No".

    ISSUP: Assess whether the response segment is supported by the retrieved document. A response is supported if it directly uses information 
    from the document or aligns with its content. Answer with "Yes" or "No".

    Here are the inputs:

    User Query: {query}

    1. Retrieved Context: {contexts[0]}
       Generated Response: {responses[0]}
       ISSUP: 
       ISUSE: 

    2. Retrieved Context: {contexts[1]}
       Generated Response: {responses[1]}
       ISSUP: 
       ISUSE: 

    3. Retrieved Context: {contexts[2]}
       Generated Response: {responses[2]}
       ISSUP: 
       ISUSE: 
    """
    inputs = tokenizer(critique_prompt, return_tensors="pt")
    inputs = inputs.to('cuda')
    outputs = model.generate(**inputs, max_length=2048)
    critique_result = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract ISSUP and ISUSE for each response
    results = []
    for i in range(3):
        issup = "Yes" in critique_result.split(f"{i+1}.  Retrieved Context:")[1].split("ISSUP:")[-1].split("ISUSE:")[0]
        isuse = "Yes" in critique_result.split(f"{i+1}.  Retrieved Context:")[1].split("ISUSE:")[-1]
        results.append((issup, isuse))

    return results

In [52]:
def retrieve_context(retriever, query):
    return retriever.get_relevant_documents(query)

# Generate predictions (next output segment)
def generate_response(prompt):
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = inputs.to('cuda')
    input_token = len(tokenizer(prompt)["input_ids"])
    outputs = model.generate(**inputs, max_length=input_token + 100)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split('Answer:')[1]

# Main Self-RAG algorithm
def self_rag_inference(query, retriever, max_generations=3):
    response = ""
    for t in range(max_generations):
        # Retrieve relevant text passages
        relevant_contexts = retrieve_context(retriever, query)
        # Generate responses for each context
        responses = [generate_response(f"Context: {context.page_content}\nQuery: {query}\nAnswer:") for context in relevant_contexts]
        # Critique responses
        critiques = critique_responses(query, relevant_contexts, responses)
        # Select the best response based on critique scores
        best_response, best_score = None, -1
        for i, (issup, isuse) in enumerate(critiques):
            score = int(issup) + int(isuse)  # Convert boolean to integer for scoring
            if score > best_score:
                best_response, best_score = responses[i], score
        response = best_response
    return response

In [ ]:
answers = []
for q in tqdm(question['query']):
    response = self_rag_inference(q, retriever)
    answers.append(response)

In [63]:
while len(answers) < 27:
    answers.append(None)
question['response'] = answers

In [12]:
input_tokens = []
output_tokens = []
answers = []
times = []

In [17]:
for index, row in tqdm(question_df['query'].iterrows()):
    q = row['query']
    retrieved_docs = retriever.get_relevant_documents(user_message)
    contexts = []
    for context in retrieved_docs:
        contexts.append(context.page_content)
    prompt = prompt_rag(q,contexts)
    input_token, output_token, answer, time = ask_question_with_rag(prompt)
    input_tokens.append(input_token)
    output_tokens.append(output_token)
    answers.append(answer)
    times.append(time)
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

AttributeError: 'Series' object has no attribute 'iterrows'

In [14]:
context['input_token'] = input_tokens
context['output_token'] = output_tokens
context['first_answer'] = answers
context['first_time'] = times

In [11]:
context.to_excel('Result/yi/Yi-1.5-9B-0328.xlsx', index=False)

In [11]:
input_tokens = []
output_tokens = []
answers = []
times = []
for index, row in tqdm(context[['query', 'retrived_text']].iterrows()):
    q = row['query']
    r = row['retrived_text']
    prompt = prompt_no_rag(q)
    input_token, output_token, answer, time = ask_question_with_rag_allow_exist(prompt)
    input_tokens.append(input_token)
    output_tokens.append(output_token)
    answers.append(answer)
    times.append(time)
    torch.cuda.empty_cache()

0it [00:00, ?it/s]Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
1it [00:12, 12.13s/it]Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
2it [00:23, 11.97s/it]Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
3it [00:35, 11.86s/it]Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
4it [00:47, 11.79s/it]Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
5it [00:59, 11.83s/it]Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
6it [01:11, 11.85s/it]Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
7it [01:22, 11.81s/it]Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
8it [01:34, 11.76s/it]Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
9it [01:46, 11.80s/it]Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
10it [01:58, 11.81s/it]Setting `pad_token_id` to `eos_token_id`:None for o

In [13]:
context['input_token'] = input_tokens
context['output_token'] = output_tokens
context['first_answer'] = answers
context['first_time'] = times

In [14]:
context.to_excel('Result/llama 3.2/Llama-3.1-8B-Instruct-Baseline-0519.xlsx', index=False)

In [11]:
print(answers[14])

     信贷风险是指由于不利事件或行动而导致某项业务或产品出现亏损的机会率及幅度。在评估和计算每类潜在风险后评定。例如，如果特定分类贷款所占水平相当高，导致贷款组合的资产质量下降至低于令人满意的评级，其潜在信贷风险很可能会被评定为较高的水平。信贷风险是需要金融机构识别、计算、监督和控制的重要风险之一。金融机构应设立有效的信用审批和审查部门，并制定信贷风险评估机制来评估有关信贷风险所设定的贷款限额、贷款指南、业务政策及承销标准的遵守情况。()
```


In [ ]:
output_tokens

In [ ]:
times

In [10]:
def ask_question_with_rig(prompt):
    import time
    start_time = time.time()
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = inputs.to('cuda')
    input_token = len(tokenizer(prompt)["input_ids"])
    outputs = model.generate(**inputs,max_length=input_token+300)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = response.split('Answer:\n')[1]
    output_token = len(tokenizer(answer)["input_ids"])
    time = time.time() - start_time
    return input_token, output_token, answer, time

In [9]:
input_tokens = []
output_tokens = []
retrieve_context = []
model_response = []
times = []
for _, row in tqdm(question[['query']].iterrows()):
    start_time = time.time()
    query = row['query']
    every_retrieve_context = []
    every_input_tokens = []
    every_output_tokens = []
    every_response = []
    every_times = []
    target = query
    for i in range(3):
        context = retriever.get_relevant_documents(target)
        cleaned_context = []
        for item in context:
            text = clean_text(item.page_content)
            cleaned_context.append(text)
        every_retrieve_context.append(cleaned_context)
        prompt = prompt_rag(cleaned_context,query)
        inputs = tokenizer(prompt, return_tensors="pt")
        inputs = inputs.to('cuda')
        input_token = len(tokenizer(prompt)["input_ids"])
        output = model.generate(**inputs, max_new_tokens=200)
        response = tokenizer.decode(output[0], skip_special_tokens=True)
        target = response.split('Answer:\n')[1]
        input_token_count = len(inputs['input_ids'][0])
        every_input_tokens.append(input_token_count)
        output_token_count = len(tokenizer(target)["input_ids"])
        every_output_tokens.append(output_token_count)
        every_response.append(target)
        time_use = time.time() - start_time
        every_times.append(time_use)
    retrieve_context.append(every_retrieve_context)
    input_tokens.append(every_input_tokens)
    output_tokens.append(every_output_tokens)
    model_response.append(every_response)
    times.append(every_times)
    torch.cuda.empty_cache()

0it [00:00, ?it/s]/tmp/ipykernel_207915/2017964364.py:16: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  context = retriever.get_relevant_documents(target)
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
1it [00:54, 54.32s/it]Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
2it [01:50, 55.28s/it]Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
3it [02:59, 61.61s/it]Setting 

In [11]:
input_token_df = pd.DataFrame(input_tokens, columns=['input_token_1', 'input_token_2', 'input_token_3'])
output_token_df = pd.DataFrame(output_tokens, columns=['output_token_1', 'output_token_2', 'output_token_3'])
retrieve_context_df = pd.DataFrame(retrieve_context, columns=['retrieve_context_1', 'retrieve_context_2', 'retrieve_context_3'])
model_response_df = pd.DataFrame(model_response, columns=['model_response_1', 'model_response_2', 'model_response_3'])
times_df = pd.DataFrame(times, columns=['time_1', 'time_2', 'time_3'])


final_df = pd.concat([question, input_token_df],axis = 1)
final_df = pd.concat([final_df, output_token_df],axis = 1)
final_df = pd.concat([final_df, retrieve_context_df],axis = 1)
final_df = pd.concat([final_df, model_response_df],axis = 1)
final_df = pd.concat([final_df, times_df],axis = 1)

In [11]:
input_tokens = []
output_tokens = []
answers = []
prompts = []
times = []
for _, row in tqdm(question[['query']].iterrows()):
    import time
    start_time = time.time()
    query = row['query']
    answer = query
    every_prompts = []
    every_input_tokens = []
    every_output_tokens = []
    for i in range(3):
        context = retriever.get_relevant_documents(answer)
        cleaned_context = []
        for item in context:
            text = clean_text(item.page_content)
            cleaned_context.append(text)
        prompt = prompt_rag(cleaned_context,query)
        every_prompts.append(prompt)
        inputs = tokenizer(prompt, return_tensors="pt")
        inputs = inputs.to('cuda')
        input_token = len(tokenizer(prompt)["input_ids"])
        every_input_tokens.append(input_token)
        outputs = model.generate(**inputs,max_length=input_token+100)
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        answer = response.split('Answer:\n')[1]
        output_token = len(tokenizer(answer)["input_ids"])
        every_output_tokens.append(output_token)
        time_use = time.time() - start_time
    prompts.append(every_prompts)
    input_tokens.append(every_input_tokens)
    output_tokens.append(every_output_tokens)
    answers.append(answer)
    times.append(time_use)
    torch.cuda.empty_cache()

0it [00:00, ?it/s]/tmp/ipykernel_152991/2084856173.py:15: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  context = retriever.get_relevant_documents(answer)
0it [00:03, ?it/s]
../aten/src/ATen/native/cuda/TensorCompare.cu:110: _assert_async_cuda_kernel: block: [0,0,0], thread: [0,0,0] Assertion `probability tensor contains either `inf`, `nan` or element < 0` failed.


RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [70]:
torch.cuda.empty_cache()

In [24]:
input_token_df = pd.DataFrame(input_tokens, columns=['input_token_1', 'input_token_2', 'input_token_3'])
output_token_df = pd.DataFrame(output_tokens, columns=['output_token_1', 'output_token_2', 'output_token_3'])
prompt_df = pd.DataFrame(prompts, columns=['prompt_1', 'prompt_2', 'prompt_3'])

question['response'] = answers
question['time'] = times

final_df = pd.concat([question, input_token_df],axis = 1)
final_df = pd.concat([final_df, output_token_df],axis = 1)
final_df = pd.concat([final_df, prompt_df],axis = 1)

In [13]:
final_df.to_excel('Result/RIG/llama-3.1-8b-it-2927.xlsx', index=False)

In [ ]:
final_df

In [64]:
question.to_excel('Result/self-rag/try.xlsx', index=False)